# Run: dental clinic, train split, GLM-4.7 with thinking on

Both arms (single-shot and agentic) on the clinic's **train** questions, judged by GLM-4.7. The clinic test split stays untouched until the P8 freeze.

Starts as a smoke test of the Colab path: `LIMIT = 2` questions per model (arc + str), so 8 cells, a few minutes and well under $0.50. To finish the run, set `LIMIT = None` in cell 5 and re-run it: finished cells are skipped.

1. **Runtime:** a standard CPU runtime. The backbone and the judge run on OpenRouter, so a GPU buys nothing here.
2. **Colab Secrets** (key icon): add `OPENROUTER_API_KEY` and switch on its *Notebook access*.
3. **Runtime → Run all.** Approve the Google Drive prompt in cell 3: graph snapshots and results are kept on Drive (`MyDrive/chatdt/`).

In [ ]:
# 1. Code + secrets
import os, subprocess
from google.colab import userdata

BRANCH = 'journal/agentic-cfg'
REPO = 'github.com/IvansSmirnoff/Chat.DT.git'

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret('GITHUB_TOKEN')
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
if not os.path.isdir('/content/Chat.DT'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, '/content/Chat.DT'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Chat.DT', 'pull', '-q', '--ff-only'], check=True)
os.chdir('/content/Chat.DT')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

os.environ['OPENROUTER_API_KEY'] = secret('OPENROUTER_API_KEY') or ''
assert os.environ['OPENROUTER_API_KEY'], 'Add OPENROUTER_API_KEY to Colab Secrets'
os.environ.update({
    'NEO4J_URI': 'bolt://127.0.0.1:7687', 'NEO4J_USER': 'neo4j', 'NEO4J_PASSWORD': 'password',
    'IFCBENCH_MANIFEST_PATH': '/content/Chat.DT/data/ifcbench/manifest.yaml',
})

In [ ]:
# 2. Dependencies + Neo4j (~2 min on a fresh VM; a re-run only restarts Neo4j)
!pip install -q -r requirements-base.txt
# Heap scales with the runtime: high-RAM runtimes can ETL the big models (sixty5 is 343 MB).
import psutil
ram_gb = psutil.virtual_memory().total / 2**30
os.environ['NEO4J_HEAP_MAX'] = '16g' if ram_gb > 60 else '4g'
os.environ['NEO4J_PAGECACHE'] = '8g' if ram_gb > 60 else '2g'
print(f'RAM {ram_gb:.0f} GB -> heap {os.environ["NEO4J_HEAP_MAX"]}')
!bash scripts/colab/bootstrap.sh

In [ ]:
# 3. Persist snapshots + results on Drive (resume across sessions)
from google.colab import drive
drive.mount('/content/drive')
PERSIST = '/content/drive/MyDrive/chatdt'
for local, remote in [('data/ifcbench/cache', 'ifcbench/cache'),
                      ('results/ifcbench/runs', 'ifcbench/runs')]:
    os.makedirs(f'{PERSIST}/{remote}', exist_ok=True)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if not os.path.islink(local):
        subprocess.run(['rm', '-rf', local])
        os.symlink(f'{PERSIST}/{remote}', local)
    print(local, '->', os.readlink(local))

In [ ]:
# 4. IFC files from HuggingFace (pinned dataset commit, sha256-verified).
#    --project dental_clinic for Tier A; --split test for all 31 test-split files (1.34 GB).
!python scripts/ifcbench/fetch.py --project dental_clinic

In [ ]:
# 5. Run. Missing snapshots are built here (ETL + export) the first time, then restored.
LLM = 'openrouter:z-ai/glm-4.7'   # dev backbone
PROVIDER = 'z-ai'                  # pin the OpenRouter provider; unpinned calls drift between providers
REASONING = 'on'                   # run factor: on | off (same in both arms)
LIMIT = 2                          # questions per model; None = all 16 train questions (resumes this run)
RUN = 'colab_clinic_train_glm47_think'
limit_arg = f'--limit {LIMIT}' if LIMIT else ''
!python -m src.agent.run --model-id dental_clinic__arc --model-id dental_clinic__str \
    --split train --llm {LLM} --provider {PROVIDER} --reasoning {REASONING} \
    --judge-provider z-ai --run {RUN} --workers 6 {limit_arg}

In [ ]:
# 6. Inspect: per-question verdicts and the judge's reasons
import json, pandas as pd
rows = [json.loads(l) for l in open(f'results/ifcbench/runs/{RUN}/episodes.jsonl')]
df = pd.DataFrame([{
    'model': r['model_id'], 'q': r['question_id'], 'cat': r['category'], 'paradigm': r['paradigm'],
    'verdict': (r['judgement'] or {}).get('verdict'), 'queries': r['episode']['n_queries'],
    'cost': round(r['episode']['cost_usd'], 4), 'question': r['question'][:70],
    'why': (r['judgement'] or {}).get('justification', '')[:120],
} for r in rows])
display(df.pivot_table(index='paradigm', columns='verdict', values='q', aggfunc='count', fill_value=0))
df.sort_values(['q', 'paradigm'])